# Exercício 05 — PySpark: GroupBy e Agregações

**Tópico:** PySpark — groupBy, agg, funções de agregação

---

## Setup
Faça o upload de `vendas.csv` e `funcionarios.csv` para o DBFS.

---

## Exercício 1 — Vendas por região
Leia `vendas.csv`.  
Agrupe por `regiao` e calcule:
- Total de vendas (`sum` de `valor * quantidade`)
- Número de transações (`count`)
- Ticket médio (`avg` de `valor`)

Ordene pelo total de vendas de forma decrescente.

---

## Exercício 2 — Top produtos por categoria
Agrupe por `categoria` e `produto` e calcule a quantidade total vendida (`sum` de `quantidade`).  
Exiba o produto mais vendido de cada categoria.

---

## Exercício 3 — Desempenho por vendedor
Agrupe por `vendedor` e calcule:
- Receita total gerada
- Maior venda única (`max` de `valor`)
- Menor venda única (`min` de `valor`)
- Número de vendas

---

## Exercício 4 — Vendas mensais
Extraia o mês da coluna `data` (use `month()` de `pyspark.sql.functions`).  
Agrupe por mês e calcule a receita total.  
Qual foi o mês com maior faturamento?

---

## Exercício 5 — Salário médio por departamento
Leia `funcionarios.csv`.  
Agrupe por `departamento` e calcule:
- Salário médio
- Salário máximo
- Número de funcionários

Filtre apenas departamentos com mais de 2 funcionários.


In [0]:
VENDAS_PATH       = "/Workspace/Users/leocodedev@outlook.com/pyspark/notebooks/exercise/data/vendas.csv"
FUNCIONARIOS_PATH = "/Workspace/Users/leocodedev@outlook.com/pyspark/notebooks/exercise/data/funcionarios.csv"

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
# Exercício 1

df1 = spark.read.csv(VENDAS_PATH, header=True, inferSchema=True)
df1 = df1.groupBy(col("regiao")).agg(sum(col("valor")*col("quantidade")).alias("total_vendas"), avg("valor").alias("ticket_medio"),count("*").alias("total_transacoes")).orderBy(col("total_vendas").desc())
display(df1)


In [0]:
# Exercício 2
df2 = spark.read.csv(VENDAS_PATH, header=True, inferSchema=True)
df2 = df2.groupBy("categoria","produto").agg(sum(col("quantidade")).alias("quantidade_total_vendida")).orderBy("quantidade_total_vendida")
display(df2)
window = Window.partitionBy("categoria").orderBy(desc("quantidade_total_vendida"))
df2 = (
    df2
    .withColumn("rank_vendas", row_number().over(window))
    .filter(col("rank_vendas") == 1)
    .drop("rank_vendas")
)
display(df2)



In [0]:
# Exercício 3

df3 = spark.read.csv(VENDAS_PATH, header=True, inferSchema=True)
df3 = df3.groupBy(col("vendedor")).agg(sum(col("valor")*col("quantidade")).alias("receita_total_gerada"), max(col("valor")).alias("maior_venda"), min(col("valor")).alias("menor_venda"), count("*").alias("total_vendas")).display()


In [0]:
# Exercício 4
df4 = spark.read.csv(VENDAS_PATH, header=True, inferSchema=True)
df4 = df4.withColumn("mes_numerico", month(col("data")))
df4 = df4.groupBy(col("mes_numerico")).agg(sum(col("valor") * col("quantidade")).alias("receita_total_gerada")).orderBy(col("receita_total_gerada").desc())
display(df4)


In [0]:
# Exercício 5
df5 = spark.read.csv(FUNCIONARIOS_PATH, header=True, inferSchema=True)
df5 = df5.groupBy(col("departamento")).agg(avg(col("salario")).alias("salario_medio"), max(col("salario")).alias("maior_salario"), count("*").alias("total_funcionario")).filter(col("total_funcionario") > 2)

display(df5)
